# PokeAPI Data Cleaning

## Imports

In [1]:
import json
from pathlib import Path

import pandas as pd

## Global Variables

In [2]:
DATA_PATH   = Path(Path.cwd().parent.parent, 'data')

BRONZE_PATH = Path(DATA_PATH / '1-bronze')
SILVER_PATH = Path(DATA_PATH / '2-silver')

ABILITY_PATH         = Path(BRONZE_PATH / "PokeAPI" / "ability")
ITEM_PATH            = Path(BRONZE_PATH / "PokeAPI" / "item")
MOVE_PATH            = Path(BRONZE_PATH / "PokeAPI" / "move")
NATURE_PATH          = Path(BRONZE_PATH / "PokeAPI" / "nature")
POKEMON_PATH         = Path(BRONZE_PATH / "PokeAPI" / "pokemon")
POKEMON_SPECIES_PATH = Path(BRONZE_PATH / "PokeAPI" / "pokemon-species")
TYPE_PATH            = Path(BRONZE_PATH / "PokeAPI" / "type")

SILVER_POKEAPI_PATH  = Path(SILVER_PATH / "PokeAPI")

GEN_DICT_NAME = {
    "generation-i": 1,
    "generation-ii": 2,
    "generation-iii": 3,
    "generation-iv": 4,
    "generation-v": 5,
    "generation-vi": 6,
    "generation-vii": 7,
    "generation-viii": 8,
    "generation-ix": 9,    
}

## Helpers

In [3]:
def create_dataframe(dir_path: Path) -> pd.DataFrame:
    data_list = []
    
    for file_path in dir_path.iterdir():
        with open(file_path, 'r') as f:
            data = json.load(f)
            data_list.append(data)

    return pd.DataFrame(data_list)

In [4]:
def normalize_json_column(df:pd.DataFrame, column_name: str) -> pd.DataFrame:
    try:
        new_columns = pd.json_normalize(df[column_name].to_list())
        for i in new_columns:
            df[f"{column_name}_{i}"] = new_columns[i]

        df = df.drop(columns=column_name)
        
        return df

    except KeyError:
        print(f"Non Existing Column with name: {column_name}")
        return df


In [5]:
def normalize_json_list_column(df:pd.DataFrame, column_name: str) -> pd.DataFrame:
    try:
        df = df.explode(column_name, ignore_index=True)
        df = normalize_json_column(df, column_name)

        return df
        

    except KeyError:
        print(f"Non Existing Column with name: {column_name}")
        return df

In [6]:
# PokeAPI markup "[label]{type:id}" becomes "label" (or the id without dashes when label is empty)
def clean_text(series: pd.Series) -> pd.Series:
    series = series.str.replace(r'\[([^\]]*)\]\{[^:}]+:([^}]+)\}', lambda m: m.group(1) or m.group(2).replace('-', ' '), regex=True)
    return series.str.replace(r'\s+', ' ', regex=True).str.strip()

## Creating DataFrames and Preparing the Data

### Ability:

In [7]:
df = create_dataframe(ABILITY_PATH)
df = df.set_index('id')

print(f"N of Rows: {len(df)}")
df.head()

N of Rows: 374


,name,is_main_series,generation,names,effect_entries,effect_changes,flavor_text_entries,pokemon
id,,,,,,,,
1,stench,True,"{'name': 'generation-iii', 'url': 'https://pok...","[{'name': 'あくしゅう', 'language': {'name': 'ja-hr...",[{'effect': 'Les capacités offensives de ce Po...,"[{'version_group': {'name': 'black-white', 'ur...","[{'flavor_text': 'Repousse POKéMON sauvage.', ...","[{'is_hidden': True, 'slot': 3, 'pokemon': {'n..."
2,drizzle,True,"{'name': 'generation-iii', 'url': 'https://pok...","[{'name': 'あめふらし', 'language': {'name': 'ja-hr...",[{'effect': 'La météo devient pluvieuse quand ...,[],"[{'flavor_text': 'Summons rain in battle.', 'l...","[{'is_hidden': True, 'slot': 3, 'pokemon': {'n..."
3,speed-boost,True,"{'name': 'generation-iii', 'url': 'https://pok...","[{'name': 'かそく', 'language': {'name': 'ja-hrkt...",[{'effect': 'La Vitesse de ce Pokémon augmente...,[],"[{'flavor_text': 'Améliore la VITESSE.', 'lang...","[{'is_hidden': False, 'slot': 1, 'pokemon': {'..."
4,battle-armor,True,"{'name': 'generation-iii', 'url': 'https://pok...","[{'name': 'カブトアーマー', 'language': {'name': 'ja-...",[{'effect': 'Les capacités ne peuvent pas infl...,[],[{'flavor_text': 'Bloque les coups critiques.'...,"[{'is_hidden': True, 'slot': 3, 'pokemon': {'n..."
5,sturdy,True,"{'name': 'generation-iii', 'url': 'https://pok...","[{'name': 'がんじょう', 'language': {'name': 'ja-hr...","[{'effect': 'Quand ce Pokémon a tous ses PV, n...","[{'version_group': {'name': 'black-white', 'ur...","[{'flavor_text': 'Negates 1-hit KO attacks.', ...","[{'is_hidden': False, 'slot': 2, 'pokemon': {'..."


In [8]:
df['is_main_series'].value_counts()

is_main_series
True     314
False     60
Name: count, dtype: int64

In [9]:
# Recreate for not popping errors when running this cell 2+ times
df = create_dataframe(ABILITY_PATH)

# Dropping columns that will not be used
df = df.drop(columns=['names', 'pokemon', 'flavor_text_entries'])

# WARNING: Since gen 9 is the current gen, I can drop this column easily, if you are working in a gen that is not the current gen: ADAPT THIS CODE!
df = df.drop(columns='effect_changes')

# Main Series only, deleting != True:
df = df[df['is_main_series']].reset_index(drop=True)
df = df.drop(columns='is_main_series')

df = normalize_json_column(df, 'generation')
df = df.drop(columns=['generation_url'])

effects = normalize_json_list_column(df[['id', 'effect_entries']], 'effect_entries')
effects = effects.loc[effects['effect_entries_language.name'] == 'en', ['id', 'effect_entries_effect', 'effect_entries_short_effect']]

df = df.drop(columns=['effect_entries']).merge(effects, on='id')

# Needs to be last step, otherwise filters won't work and dropping the non-main series ill also break
df = df.set_index('id')

print(f"N of Rows: {len(df)}")
df.head()

N of Rows: 314


,name,generation_name,effect_entries_effect,effect_entries_short_effect
id,,,,
1,stench,generation-iii,This Pokémon's damaging moves have a 10% chanc...,Has a 10% chance of making target Pokémon flin...
2,drizzle,generation-iii,The weather changes to rain when this Pokémon ...,Summons rain that lasts indefinitely upon ente...
3,speed-boost,generation-iii,This Pokémon's Speed rises one stage after eac...,Raises Speed one stage after each turn.
4,battle-armor,generation-iii,Moves cannot score critical hits against this ...,Protects against critical hits.
5,sturdy,generation-iii,"When this Pokémon is at full HP, any hit that ...","Prevents being KOed from full HP, leaving 1 HP..."


In [10]:
df['generation_name'].value_counts()

generation_name
generation-iii     76
generation-iv      47
generation-ix      47
generation-vii     42
generation-v       41
generation-viii    34
generation-vi      27
Name: count, dtype: int64

In [11]:
df['generation'] = df['generation_name'].map(GEN_DICT_NAME)
df = df.drop(columns='generation_name')

df = df.rename(columns={"effect_entries_effect": "effect"})
df = df.rename(columns={"effect_entries_short_effect": "short_effect"})

df['generation'].value_counts()

generation
3    76
4    47
9    47
7    42
5    41
8    34
6    27
Name: count, dtype: int64

In [12]:
df.head()

,name,effect,short_effect,generation
id,,,,
1,stench,This Pokémon's damaging moves have a 10% chanc...,Has a 10% chance of making target Pokémon flin...,3
2,drizzle,The weather changes to rain when this Pokémon ...,Summons rain that lasts indefinitely upon ente...,3
3,speed-boost,This Pokémon's Speed rises one stage after eac...,Raises Speed one stage after each turn.,3
4,battle-armor,Moves cannot score critical hits against this ...,Protects against critical hits.,3
5,sturdy,"When this Pokémon is at full HP, any hit that ...","Prevents being KOed from full HP, leaving 1 HP...",3


In [13]:
# Data quality checks
print(f"Duplicated ids:   {df.index.duplicated().sum()}")
print(f"Duplicated names: {df['name'].duplicated().sum()}")
print(f"Duplicated rows:  {df.duplicated().sum()}")

print("----------\nNulls per column:")
print(df.isna().sum())

text_cols = df.select_dtypes('str')

print("----------\nEmpty strings per column:")
print((text_cols.apply(lambda c: c.str.strip()) == '').sum())

print("----------\nTexts with newlines or markup ([...]{...}):")
print(text_cols.apply(lambda c: c.str.contains(r'\n|\f|\]\{', regex=True)).sum())

Duplicated ids:   0
Duplicated names: 0
Duplicated rows:  0
----------
Nulls per column:
name            0
effect          0
short_effect    0
generation      0
dtype: int64
----------
Empty strings per column:
name            0
effect          0
short_effect    0
dtype: int64
----------
Texts with newlines or markup ([...]{...}):
name              0
effect          131
short_effect      0
dtype: int64


In [14]:
# Plain text only: markup and newlines are removed from the effect texts, overwriting the original columns
df['effect'] = clean_text(df['effect'])
df['short_effect'] = clean_text(df['short_effect'])

print("Texts with newlines or markup after cleaning:")
print(df[['effect', 'short_effect']].apply(lambda c: c.str.contains(r'\n|\f|\]\{', regex=True)).sum())

Texts with newlines or markup after cleaning:
effect          0
short_effect    0
dtype: int64


In [15]:
# Data types:
df['generation'] = df['generation'].astype('int8')

print(df.dtypes)
print(f"----------\nIndex dtype: {df.index.dtype}")
print(f"Names not in kebab-case: {(~df['name'].str.fullmatch(r'[a-z0-9-]+')).sum()}")

name             str
effect           str
short_effect     str
generation      int8
dtype: object
----------
Index dtype: int64
Names not in kebab-case: 0


In [16]:
# Text lengths, to spot truncated or broken texts
lengths = df[['effect', 'short_effect']].apply(lambda c: c.str.len())
lengths.describe()

,effect,short_effect
count,314.000000,314.000000
mean,202.156051,64.159236
std,214.922122,28.943375
min,27.000000,15.000000
25%,83.250000,47.000000
50%,126.000000,58.500000
75%,240.750000,76.000000
max,1538.000000,228.000000


In [17]:
# Shortest and longest effects
df.loc[lengths.sort_values('effect').index].iloc[[0, 1, 2, -3, -2, -1]]

,name,effect,short_effect,generation
id,,,,
39,inner-focus,This Pokémon cannot flinch.,Prevents flinching.,3
169,fur-coat,Halves damage from physical attacks.,Halves damage from physical attacks.,6
247,ripen,Ripens Berries and doubles their effect.,Doubles the effect of berries.,8
104,mold-breaker,This Pokémon's moves completely ignore abiliti...,Bypasses targets' abilities if they could hind...,4
163,turboblaze,This Pokémon's moves completely ignore abiliti...,Bypasses targets' abilities if they could hind...,5
164,teravolt,This Pokémon's moves completely ignore abiliti...,Bypasses targets' abilities if they could hind...,5


In [18]:
SILVER_POKEAPI_PATH.mkdir(parents=True, exist_ok=True)

df.reset_index().to_parquet(SILVER_POKEAPI_PATH / 'ability.parquet', index=False)

check = pd.read_parquet(SILVER_POKEAPI_PATH / 'ability.parquet').set_index('id')
print(f"Rows: {len(check)} | Same dtypes: {check.dtypes.equals(df.dtypes)} | Same data: {check.equals(df)}")

Rows: 314 | Same dtypes: True | Same data: True


### Item:

In [19]:
df = create_dataframe(ITEM_PATH)
df = df.set_index('id')

print(f"N of Rows: {len(df)}")
df.head()

N of Rows: 2223


,name,fling_power,fling_effect,attributes,category,effect_entries,flavor_text_entries,game_indices,prices,names,held_by_pokemon,sprites,baby_trigger_for,machines
id,,,,,,,,,,,,,,
1,master-ball,NaN,None,"[{'name': 'countable', 'url': 'https://pokeapi...","{'name': 'standard-balls', 'url': 'https://pok...",[{'effect': 'Utilisé au combat : Capture un ...,[{'text': 'The best BALL that catches a POKéMO...,"[{'game_index': 1, 'generation': {'name': 'gen...","[{'purchase_price': None, 'sell_price': 0, 'cu...","[{'name': 'マスターボール', 'language': {'name': 'ja-...",[],{'default': 'https://raw.githubusercontent.com...,None,[]
2,ultra-ball,NaN,None,"[{'name': 'countable', 'url': 'https://pokeapi...","{'name': 'standard-balls', 'url': 'https://pok...",[{'effect': 'Utilisé au combat : Tente de ca...,[{'text': 'A better BALL with a higher catch r...,"[{'game_index': 2, 'generation': {'name': 'gen...","[{'purchase_price': 1200, 'sell_price': 600, '...","[{'name': 'ハイパーボール', 'language': {'name': 'ja-...",[],{'default': 'https://raw.githubusercontent.com...,None,[]
3,great-ball,NaN,None,"[{'name': 'countable', 'url': 'https://pokeapi...","{'name': 'standard-balls', 'url': 'https://pok...",[{'effect': 'Utilisé au combat : Tente de ca...,[{'text': 'A good BALL with a higher catch rat...,"[{'game_index': 3, 'generation': {'name': 'gen...","[{'purchase_price': 600, 'sell_price': 300, 'c...","[{'name': 'スーパーボール', 'language': {'name': 'ja-...",[],{'default': 'https://raw.githubusercontent.com...,None,[]
4,poke-ball,NaN,None,"[{'name': 'countable', 'url': 'https://pokeapi...","{'name': 'standard-balls', 'url': 'https://pok...",[{'effect': 'Utilisé au combat : Tente de ca...,[{'text': 'A tool used for catching wild POKéM...,"[{'game_index': 4, 'generation': {'name': 'gen...",[],"[{'name': 'モンスターボール', 'language': {'name': 'ja...",[],{'default': 'https://raw.githubusercontent.com...,None,[]
5,safari-ball,NaN,None,"[{'name': 'countable', 'url': 'https://pokeapi...","{'name': 'standard-balls', 'url': 'https://pok...",[{'effect': 'Utilisé au combat : Tente de ca...,[{'text': 'A special BALL that is used only in...,"[{'game_index': 8, 'generation': {'name': 'gen...",[],"[{'name': 'サファリボール', 'language': {'name': 'ja-...",[],{'default': 'https://raw.githubusercontent.com...,None,[]


In [20]:
# Recreate for not popping errors when running this cell 2+ times
df = create_dataframe(ITEM_PATH)

# Dropping columns that will not be used
df = df.drop(columns=['names', 'attributes', 'held_by_pokemon', 'machines', 'game_indices', 'prices', 'sprites', 'baby_trigger_for'])

# Fling effect is null for most items (json_normalize breaks on nulls), so only the name is extracted
df[['category', 'fling_effect']] = df[['category', 'fling_effect']].apply(lambda c: c.str.get('name'))

# Items without English entries are kept (left merge)
effects = normalize_json_list_column(df.loc[df['effect_entries'].map(len) > 0, ['id', 'effect_entries']], 'effect_entries')
effects = effects.loc[effects['effect_entries_language.name'] == 'en', ['id', 'effect_entries_effect', 'effect_entries_short_effect']]

# Latest English flavor text, used to fill the missing effects
flavors = normalize_json_list_column(df.loc[df['flavor_text_entries'].map(len) > 0, ['id', 'flavor_text_entries']], 'flavor_text_entries')
flavors = flavors[flavors['flavor_text_entries_language.name'] == 'en'].groupby('id', as_index=False)['flavor_text_entries_text'].last()

df = df.drop(columns=['effect_entries', 'flavor_text_entries']).merge(effects, on='id', how='left').merge(flavors, on='id', how='left')
df = df.rename(columns={'effect_entries_effect': 'effect', 'effect_entries_short_effect': 'short_effect', 'flavor_text_entries_text': 'flavor_text'})

# Needs to be last step, otherwise filters won't work
df = df.set_index('id')

print(f"N of Rows: {len(df)}")
df.head()

N of Rows: 2223


,name,fling_power,fling_effect,category,effect,short_effect,flavor_text
id,,,,,,,
1,master-ball,NaN,None,standard-balls,Used in battle\n: Catches a wild Pokémon wit...,Catches a wild Pokémon every time.,The best Poké Ball with the ultimate level of\...
2,ultra-ball,NaN,None,standard-balls,Used in battle\n: Attempts to catch a wild P...,Tries to catch a wild Pokémon. Success rate i...,An ultra-high-performance Poké Ball that provi...
3,great-ball,NaN,None,standard-balls,Used in battle\n: Attempts to catch a wild P...,Tries to catch a wild Pokémon. Success rate i...,"A good, high-performance Poké Ball that provid..."
4,poke-ball,NaN,None,standard-balls,Used in battle\n: Attempts to catch a wild P...,Tries to catch a wild Pokémon.,A device for catching wild Pokémon. It’s throw...
5,safari-ball,NaN,None,standard-balls,Used in battle\n: Attempts to catch a wild P...,Tries to catch a wild Pokémon in the Great Mar...,A special Poké Ball that was used in the Safar...


In [21]:
# Data quality checks
print(f"Duplicated ids:   {df.index.duplicated().sum()}")
print(f"Duplicated names: {df['name'].duplicated().sum()} -> {df.loc[df['name'].duplicated(), 'name'].to_list()}")
print(f"Duplicated rows:  {df.duplicated().sum()}")

print("----------\nNulls per column:")
print(df.isna().sum())

text_cols = df.select_dtypes('str')

print("----------\nEmpty strings per column:")
print((text_cols.apply(lambda c: c.str.strip()) == '').sum())

print("----------\nTexts with newlines or markup ([...]{...}):")
print(text_cols.apply(lambda c: c.str.contains(r'\n|\f|\]\{', regex=True)).sum())

# Effects written as "Used in battle\n:   ..."
definition_list = df['effect'].str.contains(r'\n:', regex=True)
print(f"Effects with a definition list: {definition_list.sum()}")

no_effect = df['effect'].isna()

print("----------")
print(f"Null effect with flavor text:    {(no_effect & df['flavor_text'].notna()).sum()}")
print(f"Null effect without flavor text: {(no_effect & df['flavor_text'].isna()).sum()}, by category:")
print(df.loc[no_effect & df['flavor_text'].isna(), 'category'].value_counts())

print("----------")
print(f"Distinct categories:        {df['category'].nunique()}")
print(f"Fling power out of 10..130: {(~df['fling_power'].dropna().between(10, 130)).sum()}")
print(f"Fling effect without power: {(df['fling_effect'].notna() & df['fling_power'].isna()).sum()}")

Duplicated ids:   0
Duplicated names: 1 -> ['roseli-berry']
Duplicated rows:  0
----------
Nulls per column:
name               0
fling_power     1551
fling_effect    2190
category           0
effect          1268
short_effect    1268
flavor_text      610
dtype: int64
----------
Empty strings per column:
name             0
effect          11
short_effect     1
flavor_text      0
dtype: int64
----------
Texts with newlines or markup ([...]{...}):
name               0
effect           303
short_effect       0
flavor_text     1582
dtype: int64
Effects with a definition list: 271
----------
Null effect with flavor text:    659
Null effect without flavor text: 609, by category:
category
tm-materials            222
all-machines            129
picnic                   81
sandwich-ingredients     59
mega-stones              45
tera-shard               18
held-items               11
event-items              10
gameplay                  8
evolution                 7
vitamins                  7
s

In [22]:
df.loc[df['name'].duplicated(keep=False), 'name']

id
723     roseli-berry
2279    roseli-berry
Name: name, dtype: str

In [23]:
df = df.sort_index()
df = df[~df['name'].duplicated(keep='first')]

# Clean text
effect_cols = ['effect', 'short_effect', 'flavor_text']
df[effect_cols] = df[effect_cols].apply(lambda c: clean_text(c.str.replace(r'\s*\n:\s*', ': ', regex=True)))
df[effect_cols] = df[effect_cols].apply(lambda c: c.mask(c.eq('')))

# Items without an English effect get the flavor text, marking where it came from
filled = df['flavor_text'] + ' (originally: null, filled from flavor_text)'
df['effect'] = df['effect'].fillna(filled)
df['short_effect'] = df['short_effect'].fillna(filled)
df = df.drop(columns='flavor_text')

print(f"Duplicated names: {df['name'].duplicated().sum()}")
print("----------\nTexts with newlines or markup after cleaning:")
print(df[['effect', 'short_effect']].apply(lambda c: c.str.contains(r'\n|\f|\]\{', regex=True)).sum())
print("----------\nEmpty strings after cleaning:")
print((df[['effect', 'short_effect']] == '').sum())
print(f"----------\nNull effects after filling: {df['effect'].isna().sum()}")

Duplicated names: 0
----------
Texts with newlines or markup after cleaning:
effect          0
short_effect    0
dtype: int64
----------
Empty strings after cleaning:
effect          0
short_effect    0
dtype: int64
----------
Null effects after filling: 608


In [24]:
# Data types:
df['fling_power'] = df['fling_power'].astype('Int16')
df[['category', 'fling_effect']] = df[['category', 'fling_effect']].astype('str')

print(df.dtypes)
print(f"----------\nIndex dtype: {df.index.dtype}")
print(f"Names not in kebab-case: {(~df['name'].str.fullmatch(r'[a-z0-9-]+')).sum()}")

name              str
fling_power     Int16
fling_effect      str
category          str
effect            str
short_effect      str
dtype: object
----------
Index dtype: int64
Names not in kebab-case: 0


In [25]:
df['category'].value_counts()

category
all-machines            338
dynamax-crystals        300
tm-materials            222
unused                  122
plot-advancement        101
mega-stones              92
gameplay                 82
picnic                   81
species-candies          80
held-items               72
sandwich-ingredients     59
vitamins                 48
evolution                40
loot                     38
all-mail                 36
z-crystals               29
data-cards               27
curry-ingredients        25
miracle-shooter          24
event-items              23
species-specific         22
type-enhancement         22
nature-mints             21
plates                   19
standard-balls           18
type-protection          18
jewels                   18
tera-shard               18
dex-completion           17
memories                 17
status-cures             14
baking-only              14
special-balls            13
healing                  13
collectibles             13
spelunking 

In [26]:
pd.crosstab(df['fling_power'], df['fling_effect'].fillna('none'))

fling_effect,badly-poison,berry-effect,burn,flinch,herb-effect,none,paralyze,poison
fling_power,,,,,,,,
10,0,25,0,0,2,181,0,0
20,0,0,0,0,0,12,0,0
30,1,0,1,2,0,176,1,0
40,0,0,0,0,0,3,0,0
50,0,0,0,0,0,23,0,0
60,0,0,0,0,0,10,0,0
70,0,0,0,0,0,11,0,1
80,0,0,0,0,0,136,0,0
85,0,0,0,0,0,6,0,0


In [27]:
df.loc[df['effect'].isna(), 'category'].value_counts()

category
tm-materials            222
all-machines            129
picnic                   81
sandwich-ingredients     59
mega-stones              45
tera-shard               18
held-items               11
event-items              10
gameplay                  8
evolution                 7
vitamins                  7
standard-balls            5
plates                    2
loot                      2
plot-advancement          1
pp-recovery               1
Name: count, dtype: int64

In [28]:
lengths = df[['effect', 'short_effect']].apply(lambda c: c.str.len())
lengths.describe()

,effect,short_effect
count,1614.000000,1614.000000
mean,118.540273,95.628872
std,75.592042,50.519120
min,7.000000,8.000000
25%,56.000000,49.000000
50%,125.000000,85.000000
75%,154.000000,152.000000
max,622.000000,202.000000


In [29]:
# Shortest and longest effects (null effects are left out)
df.loc[lengths['effect'].dropna().sort_values().index, ['name', 'effect', 'short_effect']].iloc[[0, 1, 2, -3, -2, -1]]

,name,effect,short_effect
id,,,
406,loot-sack,Unused.,Carries coal mine loot.
407,rule-book,Unused.,List of battle types and their rules.
418,red-chain,Unused.,Used to bind Palkia and Dialga.
72,red-shard,"No effect. In Diamond and Pearl, trade ten for...",No effect. Can be traded for items or moves.
73,blue-shard,"No effect. In Diamond and Pearl, trade ten for...",No effect. Can be traded for items or moves.
74,yellow-shard,"No effect. In Diamond and Pearl, trade ten for...",No effect. Can be traded for items or moves.


In [30]:
SILVER_POKEAPI_PATH.mkdir(parents=True, exist_ok=True)

df.reset_index().to_parquet(SILVER_POKEAPI_PATH / 'item.parquet', index=False)

check = pd.read_parquet(SILVER_POKEAPI_PATH / 'item.parquet').set_index('id')
print(f"Rows: {len(check)} | Same dtypes: {check.dtypes.equals(df.dtypes)} | Same data: {check.equals(df)}")

Rows: 2222 | Same dtypes: True | Same data: True


### Move:

In [31]:
df = create_dataframe(MOVE_PATH)
df = df.set_index('id')

print(f"N of Rows: {len(df)}")
df.head()

N of Rows: 937


,name,accuracy,effect_chance,pp,priority,power,contest_combos,contest_type,contest_effect,damage_class,...,meta,names,past_values,stat_changes,super_contest_effect,target,type,machines,flavor_text_entries,learned_by_pokemon
id,,,,,,,,,,,,,,,,,,,,,
1,pound,100.0,NaN,35.0,0,40.0,{'normal': {'use_before': [{'name': 'double-sl...,"{'name': 'tough', 'url': 'https://pokeapi.co/a...",{'url': 'https://pokeapi.co/api/v2/contest-eff...,"{'name': 'physical', 'url': 'https://pokeapi.c...",...,"{'ailment': {'name': 'none', 'url': 'https://p...","[{'name': 'はたく', 'language': {'name': 'ja-hrkt...",[],[],{'url': 'https://pokeapi.co/api/v2/super-conte...,"{'name': 'selected-pokemon', 'url': 'https://p...","{'name': 'normal', 'url': 'https://pokeapi.co/...",[],[{'flavor_text': 'Pounds with fore­ legs or ta...,"[{'name': 'clefairy', 'url': 'https://pokeapi...."
2,karate-chop,100.0,NaN,25.0,0,50.0,"{'normal': {'use_before': None, 'use_after': [...","{'name': 'tough', 'url': 'https://pokeapi.co/a...",{'url': 'https://pokeapi.co/api/v2/contest-eff...,"{'name': 'physical', 'url': 'https://pokeapi.c...",...,"{'ailment': {'name': 'none', 'url': 'https://p...","[{'name': 'からてチョップ', 'language': {'name': 'ja-...","[{'accuracy': None, 'power': None, 'pp': None,...",[],{'url': 'https://pokeapi.co/api/v2/super-conte...,"{'name': 'selected-pokemon', 'url': 'https://p...","{'name': 'fighting', 'url': 'https://pokeapi.c...",[],[{'flavor_text': 'Has a high criti­ cal hit ra...,"[{'name': 'mankey', 'url': 'https://pokeapi.co..."
3,double-slap,85.0,NaN,10.0,0,15.0,"{'normal': {'use_before': None, 'use_after': [...","{'name': 'tough', 'url': 'https://pokeapi.co/a...",{'url': 'https://pokeapi.co/api/v2/contest-eff...,"{'name': 'physical', 'url': 'https://pokeapi.c...",...,"{'ailment': {'name': 'none', 'url': 'https://p...","[{'name': 'おうふくビンタ', 'language': {'name': 'ja-...",[],[],{'url': 'https://pokeapi.co/api/v2/super-conte...,"{'name': 'selected-pokemon', 'url': 'https://p...","{'name': 'normal', 'url': 'https://pokeapi.co/...",[],[{'flavor_text': 'Repeatedly slaps 2-5 times.'...,"[{'name': 'clefairy', 'url': 'https://pokeapi...."
4,comet-punch,85.0,NaN,15.0,0,18.0,None,"{'name': 'tough', 'url': 'https://pokeapi.co/a...",{'url': 'https://pokeapi.co/api/v2/contest-eff...,"{'name': 'physical', 'url': 'https://pokeapi.c...",...,"{'ailment': {'name': 'none', 'url': 'https://p...","[{'name': 'れんぞくパンチ', 'language': {'name': 'ja-...",[],[],{'url': 'https://pokeapi.co/api/v2/super-conte...,"{'name': 'selected-pokemon', 'url': 'https://p...","{'name': 'normal', 'url': 'https://pokeapi.co/...",[{'machine': {'url': 'https://pokeapi.co/api/v...,[{'flavor_text': 'Repeatedly punches 2-5 times...,"[{'name': 'hitmonchan', 'url': 'https://pokeap..."
5,mega-punch,85.0,NaN,20.0,0,80.0,"{'normal': {'use_before': None, 'use_after': [...","{'name': 'tough', 'url': 'https://pokeapi.co/a...",{'url': 'https://pokeapi.co/api/v2/contest-eff...,"{'name': 'physical', 'url': 'https://pokeapi.c...",...,"{'ailment': {'name': 'none', 'url': 'https://p...","[{'name': 'メガトンパンチ', 'language': {'name': 'ja-...",[],[],{'url': 'https://pokeapi.co/api/v2/super-conte...,"{'name': 'selected-pokemon', 'url': 'https://p...","{'name': 'normal', 'url': 'https://pokeapi.co/...",[{'machine': {'url': 'https://pokeapi.co/api/v...,[{'flavor_text': 'A powerful punch thrown very...,"[{'name': 'charmander', 'url': 'https://pokeap..."


In [32]:
df.loc[df.index >= 10000, 'name']

id
10001     shadow-rush
10002    shadow-blast
10003    shadow-blitz
10004     shadow-bolt
10005    shadow-break
10006    shadow-chill
10007      shadow-end
10008     shadow-fire
10009     shadow-rave
10010    shadow-storm
10011     shadow-wave
10012     shadow-down
10013     shadow-half
10014     shadow-hold
10015     shadow-mist
10016    shadow-panic
10017     shadow-shed
10018      shadow-sky
Name: name, dtype: str

In [33]:
# Recreate for not popping errors when running this cell 2+ times
df = create_dataframe(MOVE_PATH)

# Dropping columns that will not be used
df = df.drop(columns=['names', 'learned_by_pokemon', 'machines'])
df = df.drop(columns=['contest_combos', 'contest_type', 'contest_effect', 'super_contest_effect'])

# WARNING: Since gen 9 is the current gen, I can drop this column easily, if you are working in a gen that is not the current gen: ADAPT THIS CODE!
df = df.drop(columns=['effect_changes', 'past_values'])

# Ids >= 10000 are not main series (shadow moves):
df = df[df['id'] < 10000].reset_index(drop=True)

name_cols = ['damage_class', 'type', 'target']
df[name_cols] = df[name_cols].apply(lambda c: c.str.get('name'))

df = normalize_json_column(df, 'generation')
df['generation'] = df['generation_name'].map(GEN_DICT_NAME)
df = df.drop(columns=['generation_name', 'generation_url'])

# Legends Arceus and Gen 9 moves have null meta (json_normalize breaks on nulls)
df['meta'] = df['meta'].map(lambda m: m or {})
df = normalize_json_column(df, 'meta')
df = df.drop(columns=['meta_ailment.url', 'meta_category.url'])
df.columns = df.columns.str.removeprefix('meta_').str.removesuffix('.name')

df['stat_changes'] = df['stat_changes'].map(lambda l: [{'stat': s['stat']['name'], 'change': s['change']} for s in l])

# Moves without English entries are kept (left merge), scale-shot has its English effect twice
effects = normalize_json_list_column(df.loc[df['effect_entries'].map(len) > 0, ['id', 'effect_entries']], 'effect_entries')
effects = effects.loc[effects['effect_entries_language.name'] == 'en', ['id', 'effect_entries_effect', 'effect_entries_short_effect']].drop_duplicates()

# Latest English flavor text, used to fill the missing effects
flavors = normalize_json_list_column(df.loc[df['flavor_text_entries'].map(len) > 0, ['id', 'flavor_text_entries']], 'flavor_text_entries')
flavors = flavors[flavors['flavor_text_entries_language.name'] == 'en'].groupby('id', as_index=False)['flavor_text_entries_flavor_text'].last()

df = df.drop(columns=['effect_entries', 'flavor_text_entries']).merge(effects, on='id', how='left').merge(flavors, on='id', how='left')
df = df.rename(columns={'effect_entries_effect': 'effect', 'effect_entries_short_effect': 'short_effect', 'flavor_text_entries_flavor_text': 'flavor_text'})

# Needs to be last step, otherwise filters won't work
df = df.set_index('id')

print(f"N of Rows: {len(df)}")
df.head()

N of Rows: 919


,name,accuracy,effect_chance,pp,priority,power,damage_class,stat_changes,target,type,...,healing,crit_rate,ailment_chance,flinch_chance,stat_chance,ailment,category,effect,short_effect,flavor_text
id,,,,,,,,,,,,,,,,,,,,,
1,pound,100.0,NaN,35.0,0,40.0,physical,[],selected-pokemon,normal,...,0.0,0.0,0.0,0.0,0.0,none,damage,Inflicts regular damage.,Inflicts regular damage with no additional eff...,The target is physically pounded with a long t...
2,karate-chop,100.0,NaN,25.0,0,50.0,physical,[],selected-pokemon,fighting,...,0.0,1.0,0.0,0.0,0.0,none,damage,Inflicts regular damage. User’s critical hit ...,Has an increased chance for a critical hit.,This move can’t be used.\nIt’s recommended tha...
3,double-slap,85.0,NaN,10.0,0,15.0,physical,[],selected-pokemon,normal,...,0.0,0.0,0.0,0.0,0.0,none,damage,Inflicts regular damage. Hits 2–5 times in on...,Hits 2-5 times in one turn.,This move can’t be used.\nIt’s recommended tha...
4,comet-punch,85.0,NaN,15.0,0,18.0,physical,[],selected-pokemon,normal,...,0.0,0.0,0.0,0.0,0.0,none,damage,Inflicts regular damage. Hits 2–5 times in on...,Hits 2-5 times in one turn.,This move can’t be used.\nIt’s recommended tha...
5,mega-punch,85.0,NaN,20.0,0,80.0,physical,[],selected-pokemon,normal,...,0.0,0.0,0.0,0.0,0.0,none,damage,Inflicts regular damage.,Inflicts regular damage with no additional eff...,The target is slugged by a punch thrown with m...


In [34]:
# Data quality checks
print(f"Duplicated ids:   {df.index.duplicated().sum()}")
print(f"Duplicated names: {df['name'].duplicated().sum()}")
print(f"Duplicated rows:  {df.drop(columns='stat_changes').duplicated().sum()}")

print("----------\nNulls per column:")
print(df.isna().sum())

text_cols = df.select_dtypes('str')
print("----------\nEmpty strings per column:")
print((text_cols.apply(lambda c: c.str.strip()) == '').sum())
print("----------\nTexts with newlines or markup ([...]{...}):")
print(text_cols.apply(lambda c: c.str.contains(r'\n|\f|\]\{', regex=True)).sum())

no_effect = df['effect'].isna()
print("----------")
print(f"Null effect with flavor text:    {(no_effect & df['flavor_text'].notna()).sum()}")
print(f"Null effect without flavor text: {(no_effect & df['flavor_text'].isna()).sum()} -> {df.loc[no_effect & df['flavor_text'].isna(), 'name'].to_list()}")

status_power = df['damage_class'].eq('status') & df['power'].notna()
placeholder_power = df['damage_class'].ne('status') & df['power'].isin([0, 1])
bad_accuracy = df['accuracy'].notna() & ~df['accuracy'].between(1, 100)

print("----------")
print(f"Damage classes:                 {sorted(df['damage_class'].unique())}")
print(f"Status moves with power:        {status_power.sum()} -> {df.loc[status_power, 'name'].to_list()}")
print(f"Non-status moves without power: {(df['damage_class'].ne('status') & df['power'].isna()).sum()}")
print(f"Damaging moves with power 0/1:  {placeholder_power.sum()} -> {df.loc[placeholder_power, 'name'].to_list()}")
print(f"Distinct types:                 {df['type'].nunique()}")
print(f"Accuracy out of 1..100:         {bad_accuracy.sum()} -> {df.loc[bad_accuracy, 'name'].to_list()}")
print(f"Priority out of -7..+5:         {(~df['priority'].between(-7, 5)).sum()}")
print(f"Only one of min/max hits null:  {(df['min_hits'].isna() != df['max_hits'].isna()).sum()}")
print(f"min_hits > max_hits:            {(df['min_hits'] > df['max_hits']).sum()}")

# A stage change is ±1 to ±3 on one of the 7 battle stats
changes = df['stat_changes'].explode().dropna()
valid_stats = {'attack', 'defense', 'special-attack', 'special-defense', 'speed', 'accuracy', 'evasion'}

print("----------")
print(f"Stat changes out of ±1..±3:     {(~changes.str.get('change').astype(int).abs().between(1, 3)).sum()}")
print(f"Stat changes with unknown stat: {(~changes.str.get('stat').isin(valid_stats)).sum()}")

Duplicated ids:   0
Duplicated names: 0
Duplicated rows:  0
----------
Nulls per column:
name                0
accuracy          285
effect_chance     653
pp                  0
priority            0
power             331
damage_class        0
stat_changes        0
target              0
type                0
generation          0
min_hits          891
max_hits          891
min_turns         875
max_turns         875
drain              92
healing            92
crit_rate          92
ailment_chance     92
flinch_chance      92
stat_chance        92
ailment            92
category           92
effect             93
short_effect       93
flavor_text         5
dtype: int64
----------
Empty strings per column:
name            0
ailment         0
category        0
effect          0
short_effect    0
flavor_text     0
dtype: int64
----------
Texts with newlines or markup ([...]{...}):
name              0
ailment           0
category          0
effect          204
short_effect      0
flavor_text  

In [35]:
# Plain text only: markup and newlines are removed, overwriting the original columns
effect_cols = ['effect', 'short_effect', 'flavor_text']
df[effect_cols] = df[effect_cols].apply(clean_text)

# Moves without an English effect get the flavor text, marking where it came from
filled = df['flavor_text'] + ' (originally: null, filled from flavor_text)'
df['effect'] = df['effect'].fillna(filled)
df['short_effect'] = df['short_effect'].fillna(filled)
df = df.drop(columns='flavor_text')

print("Texts with newlines or markup after cleaning:")
print(df[['effect', 'short_effect']].apply(lambda c: c.str.contains(r'\n|\f|\]\{', regex=True)).sum())
print(f"----------\nNull effects after filling: {df['effect'].isna().sum()}")

Texts with newlines or markup after cleaning:
effect          0
short_effect    0
dtype: int64
----------
Null effects after filling: 5


In [36]:
# Gen 9 entries use 0 (and 1) where older moves use null: no fixed power / never misses
print(df.loc[df['power'].isin([0, 1]) | df['accuracy'].eq(0), ['name', 'damage_class', 'power', 'accuracy']])

df['power'] = df['power'].mask(df['power'].isin([0, 1]))
df['accuracy'] = df['accuracy'].mask(df['accuracy'].eq(0))

print(f"----------\nNull power: {df['power'].isna().sum()} | Null accuracy: {df['accuracy'].isna().sum()}")
print(f"Status moves with power: {(df['damage_class'].eq('status') & df['power'].notna()).sum()}")
print(f"Accuracy out of 1..100:  {(~df['accuracy'].dropna().between(1, 100)).sum()}")

                 name damage_class  power  accuracy
id                                                 
829       power-shift       status    0.0       NaN
837     victory-dance       status    0.0       NaN
842           shelter       status    0.0       NaN
849    lunar-blessing       status    0.0       NaN
850        take-heart       status    0.0       NaN
852         silk-trap       status    0.0       NaN
858     spicy-extract       status    0.0       NaN
863  revival-blessing       status    0.0       NaN
867            doodle       status    0.0     100.0
868       fillet-away       status    0.0       NaN
877         ruination      special    1.0      90.0
880         shed-tail       status    0.0       NaN
881  chilly-reception       status    0.0       NaN
882           tidy-up       status    0.0       NaN
883         snowscape       status    0.0       NaN
894       comeuppance     physical    1.0     100.0
908   burning-bulwark       status    0.0       0.0
911    tachy

In [37]:
# Data types: nullable Int keeps the nulls (variable power, moves without meta)
int8_cols = ['accuracy', 'effect_chance', 'min_hits', 'max_hits', 'min_turns', 'max_turns', 'drain', 'healing', 'crit_rate', 'ailment_chance', 'flinch_chance', 'stat_chance']
str_cols  = ['damage_class', 'type', 'target', 'ailment', 'category']

df['power'] = df['power'].astype('Int16')
df[int8_cols] = df[int8_cols].astype('Int8')
df[['pp', 'priority', 'generation']] = df[['pp', 'priority', 'generation']].astype('int8')
df[str_cols] = df[str_cols].astype('str')

print(df.dtypes)
print(f"----------\nIndex dtype: {df.index.dtype}")
print(f"Names not in kebab-case: {(~df['name'].str.fullmatch(r'[a-z0-9-]+')).sum()}")

name                 str
accuracy            Int8
effect_chance       Int8
pp                  int8
priority            int8
power              Int16
damage_class         str
stat_changes      object
target               str
type                 str
generation          int8
min_hits            Int8
max_hits            Int8
min_turns           Int8
max_turns           Int8
drain               Int8
healing             Int8
crit_rate           Int8
ailment_chance      Int8
flinch_chance       Int8
stat_chance         Int8
ailment              str
category             str
effect               str
short_effect         str
dtype: object
----------
Index dtype: int64
Names not in kebab-case: 0


In [38]:
df.groupby('damage_class')['power'].describe()

,count,mean,std,min,25%,50%,75%,max
damage_class,,,,,,,,
physical,345.0,74.510145,39.863146,10.0,50.0,75.0,95.0,250.0
special,224.0,88.549107,36.294353,15.0,65.0,80.0,110.0,200.0
status,0.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


In [39]:
pd.crosstab(df['type'], df['damage_class'])

damage_class,physical,special,status
type,,,
bug,17,7,10
dark,30,7,15
dragon,13,15,3
electric,18,23,8
fairy,6,15,12
fighting,43,6,8
fire,18,26,3
flying,17,9,5
ghost,17,10,7


In [40]:
df['priority'].value_counts().sort_index()

priority
-7      1
-6      5
-5      2
-4      2
-3      3
-1      1
 0    863
 1     16
 2      7
 3      6
 4     12
 5      1
Name: count, dtype: int64

In [41]:
df['category'].value_counts(dropna=False)

category
damage                391
unique                106
NaN                    92
damage-ailment         84
net-good-stats         68
damage-lower           52
ailment                31
damage-raise           28
whole-field-effect     17
field-effect           15
heal                   14
damage-heal            11
ohko                    4
swagger                 4
force-switch            2
Name: count, dtype: int64

In [42]:
# Text lengths, to spot truncated or broken texts
lengths = df[['effect', 'short_effect']].apply(lambda c: c.str.len())
lengths.describe()

,effect,short_effect
count,914.000000,914.000000
mean,154.114880,65.619256
std,190.907554,41.683120
min,13.000000,12.000000
25%,59.000000,40.000000
50%,93.000000,50.000000
75%,182.750000,72.750000
max,2549.000000,215.000000


In [43]:
# Shortest and longest effects (the 5 null effects are left out)
df.loc[lengths['effect'].dropna().sort_values().index, ['name', 'effect', 'short_effect']].iloc[[0, 1, 2, -3, -2, -1]]

,name,effect,short_effect
id,,,
607,hold-hands,Does nothing.,Does nothing.
606,celebrate,Does nothing.,Does nothing.
261,will-o-wisp,Burns the target.,Burns the target.
507,sky-drop,Inflicts regular damage. User carries the targ...,"Carries the target high into the air, dodging ..."
164,substitute,Transfers 1/4 the user’s max HP into a doll th...,Transfers 1/4 of the user’s max HP into a doll...
119,mirror-move,Uses the last move targeted at the user by a P...,Uses the target’s last used move.


In [44]:
SILVER_POKEAPI_PATH.mkdir(parents=True, exist_ok=True)

df.reset_index().to_parquet(SILVER_POKEAPI_PATH / 'move.parquet', index=False)

# pyarrow reads the lists back as numpy arrays, so they are converted to lists before comparing
check = pd.read_parquet(SILVER_POKEAPI_PATH / 'move.parquet').set_index('id')
check['stat_changes'] = check['stat_changes'].map(list)
print(f"Rows: {len(check)} | Same dtypes: {check.dtypes.equals(df.dtypes)} | Same data: {check.equals(df)}")

Rows: 919 | Same dtypes: True | Same data: True


### Nature:

In [45]:
df = create_dataframe(NATURE_PATH)
df = df.set_index('id')

print(f"N of Rows: {len(df)}")
df.head()

N of Rows: 25


,name,decreased_stat,increased_stat,likes_flavor,hates_flavor,pokeathlon_stat_changes,move_battle_style_preferences,names
id,,,,,,,,
1,hardy,None,None,None,None,"[{'max_change': -1, 'pokeathlon_stat': {'name'...","[{'low_hp_preference': 61, 'high_hp_preference...","[{'name': 'がんばりや', 'language': {'name': 'ja-hr..."
2,bold,"{'name': 'attack', 'url': 'https://pokeapi.co/...","{'name': 'defense', 'url': 'https://pokeapi.co...","{'name': 'sour', 'url': 'https://pokeapi.co/ap...","{'name': 'spicy', 'url': 'https://pokeapi.co/a...","[{'max_change': -2, 'pokeathlon_stat': {'name'...","[{'low_hp_preference': 32, 'high_hp_preference...","[{'name': 'ずぶとい', 'language': {'name': 'ja-hrk..."
3,modest,"{'name': 'attack', 'url': 'https://pokeapi.co/...","{'name': 'special-attack', 'url': 'https://pok...","{'name': 'dry', 'url': 'https://pokeapi.co/api...","{'name': 'spicy', 'url': 'https://pokeapi.co/a...","[{'max_change': -2, 'pokeathlon_stat': {'name'...","[{'low_hp_preference': 34, 'high_hp_preference...","[{'name': 'ひかえめ', 'language': {'name': 'ja-hrk..."
4,calm,"{'name': 'attack', 'url': 'https://pokeapi.co/...","{'name': 'special-defense', 'url': 'https://po...","{'name': 'bitter', 'url': 'https://pokeapi.co/...","{'name': 'spicy', 'url': 'https://pokeapi.co/a...","[{'max_change': -2, 'pokeathlon_stat': {'name'...","[{'low_hp_preference': 25, 'high_hp_preference...","[{'name': 'おだやか', 'language': {'name': 'ja-hrk..."
5,timid,"{'name': 'attack', 'url': 'https://pokeapi.co/...","{'name': 'speed', 'url': 'https://pokeapi.co/a...","{'name': 'sweet', 'url': 'https://pokeapi.co/a...","{'name': 'spicy', 'url': 'https://pokeapi.co/a...","[{'max_change': 2, 'pokeathlon_stat': {'name':...","[{'low_hp_preference': 30, 'high_hp_preference...","[{'name': 'おくびょう', 'language': {'name': 'ja-hr..."


In [46]:
# Recreate for not popping errors when running this cell 2+ times
df = create_dataframe(NATURE_PATH)

df = df.drop(columns=['names', 'pokeathlon_stat_changes', 'move_battle_style_preferences'])

# Neutral natures have null stats and flavors (json_normalize breaks on nulls), so only the name is extracted
nature_cols = ['increased_stat', 'decreased_stat', 'likes_flavor', 'hates_flavor']
df[nature_cols] = df[nature_cols].apply(lambda c: c.str.get('name'))

df = df.set_index('id')

print(f"N of Rows: {len(df)}")
df.head()

N of Rows: 25


,name,decreased_stat,increased_stat,likes_flavor,hates_flavor
id,,,,,
1,hardy,None,None,None,None
2,bold,attack,defense,sour,spicy
3,modest,attack,special-attack,dry,spicy
4,calm,attack,special-defense,bitter,spicy
5,timid,attack,speed,sweet,spicy


In [47]:
df['decreased_stat'].value_counts()

decreased_stat
attack             4
defense            4
special-attack     4
special-defense    4
speed              4
Name: count, dtype: int64

In [48]:
df['likes_flavor'].value_counts()

likes_flavor
sour      4
dry       4
bitter    4
sweet     4
spicy     4
Name: count, dtype: int64

In [49]:
# Data quality checks
print(f"Duplicated ids:   {df.index.duplicated().sum()}")
print(f"Duplicated names: {df['name'].duplicated().sum()}")
print(f"Duplicated rows:  {df.duplicated().sum()}")

print("----------\nNulls per column:")
print(df.isna().sum())
print(f"Natures with all 4 null: {df[nature_cols].isna().all(axis=1).sum()}")

# A nature likes the flavor of the stat it raises and hates the flavor of the stat it lowers
stat_flavor = {'attack': 'spicy', 'defense': 'sour', 'special-attack': 'dry', 'special-defense': 'bitter', 'speed': 'sweet'}
stated = df.dropna()

print("----------")
print(f"Increased == Decreased:                {(stated['increased_stat'] == stated['decreased_stat']).sum()}")
print(f"Distinct (increased, decreased) pairs: {len(stated[['increased_stat', 'decreased_stat']].drop_duplicates())}")

Duplicated ids:   0
Duplicated names: 0
Duplicated rows:  0
----------
Nulls per column:
name              0
decreased_stat    5
increased_stat    5
likes_flavor      5
hates_flavor      5
dtype: int64
Natures with all 4 null: 5
----------
Increased == Decreased:                0
Distinct (increased, decreased) pairs: 20


In [50]:
# Data types: str keeps the neutral natures as null
df[nature_cols] = df[nature_cols].astype('str')

print(df.dtypes)
print(f"----------\nIndex dtype: {df.index.dtype}")
print(f"Names not in kebab-case: {(~df['name'].str.fullmatch(r'[a-z0-9-]+')).sum()}")
print(f"Nulls after casting: {df[nature_cols].isna().sum().sum()}")

name              str
decreased_stat    str
increased_stat    str
likes_flavor      str
hates_flavor      str
dtype: object
----------
Index dtype: int64
Names not in kebab-case: 0
Nulls after casting: 20


In [51]:
# Expected: 0 on the diagonal and 1 everywhere else
pd.crosstab(df['increased_stat'], df['decreased_stat'])

decreased_stat,attack,defense,special-attack,special-defense,speed
increased_stat,,,,,
attack,0,1,1,1,1
defense,1,0,1,1,1
special-attack,1,1,0,1,1
special-defense,1,1,1,0,1
speed,1,1,1,1,0


In [52]:
SILVER_POKEAPI_PATH.mkdir(parents=True, exist_ok=True)

df.reset_index().to_parquet(SILVER_POKEAPI_PATH / 'nature.parquet', index=False)

check = pd.read_parquet(SILVER_POKEAPI_PATH / 'nature.parquet').set_index('id')
print(f"Rows: {len(check)} | Same dtypes: {check.dtypes.equals(df.dtypes)} | Same data: {check.equals(df)}")

Rows: 25 | Same dtypes: True | Same data: True


### Type:

In [53]:
df = create_dataframe(TYPE_PATH)
df = df.set_index('id')

print(f"N of Rows: {len(df)}")
df.head()

N of Rows: 21


,name,damage_relations,past_damage_relations,game_indices,generation,move_damage_class,names,pokemon,moves,sprites
id,,,,,,,,,,
1,normal,"{'no_damage_to': [{'name': 'ghost', 'url': 'ht...",[],"[{'game_index': 0, 'generation': {'name': 'gen...","{'name': 'generation-i', 'url': 'https://pokea...","{'name': 'physical', 'url': 'https://pokeapi.c...","[{'name': 'ノーマル', 'language': {'name': 'ja-hrk...","[{'slot': 1, 'pokemon': {'name': 'pidgey', 'ur...","[{'name': 'pound', 'url': 'https://pokeapi.co/...",{'generation-v': {'black-white': {'name_icon':...
2,fighting,"{'no_damage_to': [{'name': 'ghost', 'url': 'ht...",[],"[{'game_index': 1, 'generation': {'name': 'gen...","{'name': 'generation-i', 'url': 'https://pokea...","{'name': 'physical', 'url': 'https://pokeapi.c...","[{'name': 'かくとう', 'language': {'name': 'ja-hrk...","[{'slot': 1, 'pokemon': {'name': 'mankey', 'ur...","[{'name': 'karate-chop', 'url': 'https://pokea...",{'generation-v': {'black-white': {'name_icon':...
3,flying,"{'no_damage_to': [], 'half_damage_to': [{'name...",[],"[{'game_index': 2, 'generation': {'name': 'gen...","{'name': 'generation-i', 'url': 'https://pokea...","{'name': 'physical', 'url': 'https://pokeapi.c...","[{'name': 'ひこう', 'language': {'name': 'ja-hrkt...","[{'slot': 2, 'pokemon': {'name': 'charizard', ...","[{'name': 'gust', 'url': 'https://pokeapi.co/a...",{'generation-v': {'black-white': {'name_icon':...
4,poison,"{'no_damage_to': [{'name': 'steel', 'url': 'ht...","[{'generation': {'name': 'generation-i', 'url'...","[{'game_index': 3, 'generation': {'name': 'gen...","{'name': 'generation-i', 'url': 'https://pokea...","{'name': 'physical', 'url': 'https://pokeapi.c...","[{'name': 'どく', 'language': {'name': 'ja-hrkt'...","[{'slot': 2, 'pokemon': {'name': 'bulbasaur', ...","[{'name': 'poison-sting', 'url': 'https://poke...",{'generation-v': {'black-white': {'name_icon':...
5,ground,"{'no_damage_to': [{'name': 'flying', 'url': 'h...",[],"[{'game_index': 4, 'generation': {'name': 'gen...","{'name': 'generation-i', 'url': 'https://pokea...","{'name': 'physical', 'url': 'https://pokeapi.c...","[{'name': 'じめん', 'language': {'name': 'ja-hrkt...","[{'slot': 1, 'pokemon': {'name': 'sandshrew', ...","[{'name': 'sand-attack', 'url': 'https://pokea...",{'generation-v': {'black-white': {'name_icon':...


In [54]:
df['name'].sort_index()

id
1          normal
2        fighting
3          flying
4          poison
5          ground
6            rock
7             bug
8           ghost
9           steel
10           fire
11          water
12          grass
13       electric
14        psychic
15            ice
16         dragon
17           dark
18          fairy
19        stellar
10001     unknown
10002      shadow
Name: name, dtype: str

In [55]:
# Recreate for not popping errors when running this cell 2+ times
df = create_dataframe(TYPE_PATH)

df = df.drop(columns=['names', 'game_indices', 'sprites', 'pokemon', 'moves', 'move_damage_class'])

# WARNING: Since gen 9 is the current gen, I can drop this column easily, if you are working in a gen that is not the current gen: ADAPT THIS CODE!
df = df.drop(columns='past_damage_relations')

# Ids >= 10000 are not main series (unknown, shadow):
df = df[df['id'] < 10000].reset_index(drop=True)

df = normalize_json_column(df, 'generation')
df['generation'] = df['generation_name'].map(GEN_DICT_NAME)
df = df.drop(columns=['generation_name', 'generation_url'])

# Damage relations keep only the type names, urls are dropped
to_cols   = ['no_damage_to', 'half_damage_to', 'double_damage_to']
from_cols = ['no_damage_from', 'half_damage_from', 'double_damage_from']

df = normalize_json_column(df, 'damage_relations')
df.columns = df.columns.str.removeprefix('damage_relations_')
df[to_cols + from_cols] = df[to_cols + from_cols].map(lambda l: [t['name'] for t in l])

# Needs to be last step, otherwise filters won't work
df = df.set_index('id')

print(f"N of Rows: {len(df)}")
df.head()

N of Rows: 19


,name,generation,no_damage_to,half_damage_to,double_damage_to,no_damage_from,half_damage_from,double_damage_from
id,,,,,,,,
1,normal,1,[ghost],"[rock, steel]",[],[ghost],[],[fighting]
2,fighting,1,[ghost],"[flying, poison, bug, psychic, fairy]","[normal, rock, steel, ice, dark]",[],"[rock, bug, dark]","[flying, psychic, fairy]"
3,flying,1,[],"[rock, steel, electric]","[fighting, bug, grass]",[ground],"[fighting, bug, grass]","[rock, electric, ice]"
4,poison,1,[steel],"[poison, ground, rock, ghost]","[grass, fairy]",[],"[fighting, poison, bug, grass, fairy]","[ground, psychic]"
5,ground,1,[flying],"[bug, grass]","[poison, rock, steel, fire, electric]",[electric],"[poison, rock]","[water, grass, ice]"


In [56]:
# Data quality checks
print(f"Duplicated ids:   {df.index.duplicated().sum()}")
print(f"Duplicated names: {df['name'].duplicated().sum()}")

print("----------\nNulls per column:")
print(df.isna().sum())

print("----------\nEmpty lists per column:")
print(df[to_cols + from_cols].map(len).eq(0).sum())

Duplicated ids:   0
Duplicated names: 0
----------
Nulls per column:
name                  0
generation            0
no_damage_to          0
half_damage_to        0
double_damage_to      0
no_damage_from        0
half_damage_from      0
double_damage_from    0
dtype: int64
----------
Empty lists per column:
no_damage_to          11
half_damage_to         1
double_damage_to       2
no_damage_from        12
half_damage_from       2
double_damage_from     1
dtype: int64


In [57]:
df = df.drop(columns=from_cols)

df.head()

,name,generation,no_damage_to,half_damage_to,double_damage_to
id,,,,,
1,normal,1,[ghost],"[rock, steel]",[]
2,fighting,1,[ghost],"[flying, poison, bug, psychic, fairy]","[normal, rock, steel, ice, dark]"
3,flying,1,[],"[rock, steel, electric]","[fighting, bug, grass]"
4,poison,1,[steel],"[poison, ground, rock, ghost]","[grass, fairy]"
5,ground,1,[flying],"[bug, grass]","[poison, rock, steel, fire, electric]"


In [58]:
# Data types:
df['generation'] = df['generation'].astype('int8')

print(df.dtypes)
print(f"----------\nIndex dtype: {df.index.dtype}")
print(f"Names not in kebab-case: {(~df['name'].str.fullmatch(r'[a-z0-9-]+')).sum()}")
print(f"Relations that are not lists: {(~df[to_cols].map(lambda l: isinstance(l, list))).sum().sum()}")

name                   str
generation            int8
no_damage_to        object
half_damage_to      object
double_damage_to    object
dtype: object
----------
Index dtype: int64
Names not in kebab-case: 0
Relations that are not lists: 0


In [59]:
df['generation'].value_counts().sort_index()

generation
1    15
2     2
6     1
9     1
Name: count, dtype: int64

In [60]:
lengths = df[to_cols].map(len)
lengths.describe()

,no_damage_to,half_damage_to,double_damage_to
count,19.000000,19.000000,19.000000
mean,0.421053,3.210526,2.684211
std,0.507257,1.812884,1.416280
min,0.000000,0.000000,0.000000
25%,0.000000,2.000000,2.000000
50%,0.000000,3.000000,3.000000
75%,1.000000,4.000000,3.500000
max,1.000000,7.000000,5.000000


In [61]:
# All types sorted by double damage
pd.concat([df['name'], lengths], axis=1).sort_values('double_damage_to')

,name,no_damage_to,half_damage_to,double_damage_to
id,,,,
1,normal,1,2,0
19,stellar,0,0,0
16,dragon,1,1,1
8,ghost,1,1,2
13,electric,1,3,2
14,psychic,1,2,2
17,dark,0,3,2
4,poison,1,4,2
11,water,0,3,3


In [62]:
SILVER_POKEAPI_PATH.mkdir(parents=True, exist_ok=True)

df.reset_index().to_parquet(SILVER_POKEAPI_PATH / 'type.parquet', index=False)

# pyarrow reads the lists back as numpy arrays, so they are converted to lists before comparing
check = pd.read_parquet(SILVER_POKEAPI_PATH / 'type.parquet').set_index('id')
check[to_cols] = check[to_cols].map(list)
print(f"Rows: {len(check)} | Same dtypes: {check.dtypes.equals(df.dtypes)} | Same data: {check.equals(df)}")

Rows: 19 | Same dtypes: True | Same data: True


### Pokemon + Pokemon_Species:

In [63]:
df_species = create_dataframe(POKEMON_SPECIES_PATH)

df_species = df_species[['id', 'is_legendary', 'is_mythical', 'generation']].rename(columns={'id': 'species_id'})
df_species = normalize_json_column(df_species, 'generation')
df_species = df_species.drop(columns='generation_url')
df_species['generation'] = df_species['generation_name'].map(GEN_DICT_NAME)
df_species = df_species.drop(columns='generation_name')

print(f"N of Rows: {len(df_species)}")
df_species.head()

N of Rows: 1025


,species_id,is_legendary,is_mythical,generation
0,1,False,False,1
1,2,False,False,1
2,3,False,False,1
3,4,False,False,1
4,5,False,False,1


In [64]:
df_pokemon = create_dataframe(POKEMON_PATH)
df_pokemon['species_id'] = df_pokemon['species'].str.get('url').str.extract(r'/(\d+)/$')[0].astype(int)
df_pokemon = df_pokemon.drop(columns='species')

print(f"N of Rows: {len(df_pokemon)}")
df_pokemon.head()

N of Rows: 1351


,id,name,base_experience,height,is_default,order,weight,abilities,past_abilities,forms,...,held_items,location_area_encounters,moves,sprites,cries,stats,past_stats,types,past_types,species_id
0,1,bulbasaur,64.0,7,True,1,69,"[{'is_hidden': False, 'slot': 1, 'ability': {'...","[{'generation': {'name': 'generation-iv', 'url...","[{'name': 'bulbasaur', 'url': 'https://pokeapi...",...,[],https://pokeapi.co/api/v2/pokemon/1/encounters,"[{'move': {'name': 'razor-wind', 'url': 'https...",{'other': {'home': {'front_shiny': 'https://ra...,{'latest': 'https://raw.githubusercontent.com/...,"[{'base_stat': 45, 'effort': 0, 'stat': {'name...","[{'generation': {'name': 'generation-i', 'url'...","[{'slot': 1, 'type': {'name': 'grass', 'url': ...",[],1
1,2,ivysaur,142.0,10,True,2,130,"[{'is_hidden': False, 'slot': 1, 'ability': {'...","[{'generation': {'name': 'generation-iv', 'url...","[{'name': 'ivysaur', 'url': 'https://pokeapi.c...",...,[],https://pokeapi.co/api/v2/pokemon/2/encounters,"[{'move': {'name': 'swords-dance', 'url': 'htt...",{'other': {'home': {'front_shiny': 'https://ra...,{'latest': 'https://raw.githubusercontent.com/...,"[{'base_stat': 60, 'effort': 0, 'stat': {'name...","[{'generation': {'name': 'generation-i', 'url'...","[{'slot': 1, 'type': {'name': 'grass', 'url': ...",[],2
2,3,venusaur,236.0,20,True,3,1000,"[{'is_hidden': False, 'slot': 1, 'ability': {'...","[{'generation': {'name': 'generation-iv', 'url...","[{'name': 'venusaur', 'url': 'https://pokeapi....",...,[],https://pokeapi.co/api/v2/pokemon/3/encounters,"[{'move': {'name': 'swords-dance', 'url': 'htt...",{'other': {'home': {'front_shiny': 'https://ra...,{'latest': 'https://raw.githubusercontent.com/...,"[{'base_stat': 80, 'effort': 0, 'stat': {'name...","[{'generation': {'name': 'generation-i', 'url'...","[{'slot': 1, 'type': {'name': 'grass', 'url': ...",[],3
3,4,charmander,62.0,6,True,5,85,"[{'is_hidden': False, 'slot': 1, 'ability': {'...","[{'generation': {'name': 'generation-iv', 'url...","[{'name': 'charmander', 'url': 'https://pokeap...",...,[],https://pokeapi.co/api/v2/pokemon/4/encounters,"[{'move': {'name': 'mega-punch', 'url': 'https...",{'other': {'home': {'front_shiny': 'https://ra...,{'latest': 'https://raw.githubusercontent.com/...,"[{'base_stat': 39, 'effort': 0, 'stat': {'name...","[{'generation': {'name': 'generation-i', 'url'...","[{'slot': 1, 'type': {'name': 'fire', 'url': '...",[],4
4,5,charmeleon,142.0,11,True,6,190,"[{'is_hidden': False, 'slot': 1, 'ability': {'...","[{'generation': {'name': 'generation-iv', 'url...","[{'name': 'charmeleon', 'url': 'https://pokeap...",...,[],https://pokeapi.co/api/v2/pokemon/5/encounters,"[{'move': {'name': 'mega-punch', 'url': 'https...",{'other': {'home': {'front_shiny': 'https://ra...,{'latest': 'https://raw.githubusercontent.com/...,"[{'base_stat': 58, 'effort': 0, 'stat': {'name...","[{'generation': {'name': 'generation-i', 'url'...","[{'slot': 1, 'type': {'name': 'fire', 'url': '...",[],5


In [65]:
df = df_pokemon.merge(df_species, on='species_id', how='left', validate='many_to_one')

print(f"N of Rows: {len(df)}")
df.head()

N of Rows: 1351


,id,name,base_experience,height,is_default,order,weight,abilities,past_abilities,forms,...,sprites,cries,stats,past_stats,types,past_types,species_id,is_legendary,is_mythical,generation
0,1,bulbasaur,64.0,7,True,1,69,"[{'is_hidden': False, 'slot': 1, 'ability': {'...","[{'generation': {'name': 'generation-iv', 'url...","[{'name': 'bulbasaur', 'url': 'https://pokeapi...",...,{'other': {'home': {'front_shiny': 'https://ra...,{'latest': 'https://raw.githubusercontent.com/...,"[{'base_stat': 45, 'effort': 0, 'stat': {'name...","[{'generation': {'name': 'generation-i', 'url'...","[{'slot': 1, 'type': {'name': 'grass', 'url': ...",[],1,False,False,1
1,2,ivysaur,142.0,10,True,2,130,"[{'is_hidden': False, 'slot': 1, 'ability': {'...","[{'generation': {'name': 'generation-iv', 'url...","[{'name': 'ivysaur', 'url': 'https://pokeapi.c...",...,{'other': {'home': {'front_shiny': 'https://ra...,{'latest': 'https://raw.githubusercontent.com/...,"[{'base_stat': 60, 'effort': 0, 'stat': {'name...","[{'generation': {'name': 'generation-i', 'url'...","[{'slot': 1, 'type': {'name': 'grass', 'url': ...",[],2,False,False,1
2,3,venusaur,236.0,20,True,3,1000,"[{'is_hidden': False, 'slot': 1, 'ability': {'...","[{'generation': {'name': 'generation-iv', 'url...","[{'name': 'venusaur', 'url': 'https://pokeapi....",...,{'other': {'home': {'front_shiny': 'https://ra...,{'latest': 'https://raw.githubusercontent.com/...,"[{'base_stat': 80, 'effort': 0, 'stat': {'name...","[{'generation': {'name': 'generation-i', 'url'...","[{'slot': 1, 'type': {'name': 'grass', 'url': ...",[],3,False,False,1
3,4,charmander,62.0,6,True,5,85,"[{'is_hidden': False, 'slot': 1, 'ability': {'...","[{'generation': {'name': 'generation-iv', 'url...","[{'name': 'charmander', 'url': 'https://pokeap...",...,{'other': {'home': {'front_shiny': 'https://ra...,{'latest': 'https://raw.githubusercontent.com/...,"[{'base_stat': 39, 'effort': 0, 'stat': {'name...","[{'generation': {'name': 'generation-i', 'url'...","[{'slot': 1, 'type': {'name': 'fire', 'url': '...",[],4,False,False,1
4,5,charmeleon,142.0,11,True,6,190,"[{'is_hidden': False, 'slot': 1, 'ability': {'...","[{'generation': {'name': 'generation-iv', 'url...","[{'name': 'charmeleon', 'url': 'https://pokeap...",...,{'other': {'home': {'front_shiny': 'https://ra...,{'latest': 'https://raw.githubusercontent.com/...,"[{'base_stat': 58, 'effort': 0, 'stat': {'name...","[{'generation': {'name': 'generation-i', 'url'...","[{'slot': 1, 'type': {'name': 'fire', 'url': '...",[],5,False,False,1


In [66]:
pd.crosstab(df['is_default'], df['id'] >= 10000)

id,False,True
is_default,,
False,0,326
True,1025,0


In [67]:
# Recreate for not popping errors when running this cell 2+ times
df_species = create_dataframe(POKEMON_SPECIES_PATH)

df_species = df_species[['id', 'name', 'is_legendary', 'is_mythical', 'generation', 'evolves_from_species']].rename(columns={'id': 'species_id'})
df_species['generation'] = df_species['generation'].str.get('name').map(GEN_DICT_NAME)
df_species['evolves_from_species'] = df_species['evolves_from_species'].str.get('name')

# A species is fully evolved when no other species evolves from it
fully_evolved = {}
for name, evolves_from in zip(df_species['name'], df_species['evolves_from_species']):
    fully_evolved.setdefault(name, True)
    if pd.notna(evolves_from):
        fully_evolved[evolves_from] = False

df_species['is_fully_evolved'] = df_species['name'].map(fully_evolved)
df_species = df_species.drop(columns=['name', 'evolves_from_species'])

df_pokemon = create_dataframe(POKEMON_PATH)

# Dropping columns that will not be used
df_pokemon = df_pokemon.drop(columns=['base_experience', 'height', 'order', 'forms', 'game_indices', 'held_items', 'location_area_encounters', 'sprites', 'cries'])

# WARNING: Since gen 9 is the current gen, I can drop this column easily, if you are working in a gen that is not the current gen: ADAPT THIS CODE!
df_pokemon = df_pokemon.drop(columns=['past_abilities', 'past_stats', 'past_types'])

# No row filter: ids >= 10000 are alternate forms (regional forms, Ogerpon masks, megas...), not non-main series

df_pokemon['species_id'] = df_pokemon['species'].str.get('url').str.extract(r'/(\d+)/$')[0].astype(int)
df_pokemon = df_pokemon.drop(columns='species')

# Base stats in fixed columns, EV yield (effort) is dropped
stat_cols = ['hp', 'attack', 'defense', 'special_attack', 'special_defense', 'speed']
stats = df_pokemon['stats'].map(lambda l: {s['stat']['name'].replace('-', '_'): s['base_stat'] for s in l})
df_pokemon = df_pokemon.join(pd.DataFrame(stats.to_list(), index=df_pokemon.index, columns=stat_cols))

type_cols = ['type_1', 'type_2']
types = df_pokemon['types'].map(lambda l: {f"type_{t['slot']}": t['type']['name'] for t in l})
df_pokemon = df_pokemon.join(pd.DataFrame(types.to_list(), index=df_pokemon.index, columns=type_cols))

# Keyed on is_hidden, so a hidden ability outside slot 3 would show up as an extra column
abilities = df_pokemon['abilities'].map(lambda l: {('hidden_ability' if a['is_hidden'] else f"ability_{a['slot']}"): a['ability']['name'] for a in l})
df_pokemon = df_pokemon.join(pd.DataFrame(abilities.to_list(), index=df_pokemon.index).sort_index(axis=1))

# Scarlet/Violet learnset only (Gen 9), any learn method
df_pokemon['moves'] = df_pokemon['moves'].map(lambda l: [m['move']['name'] for m in l if any(d['version_group']['name'] == 'scarlet-violet' for d in m['version_group_details'])])

df_pokemon = df_pokemon.drop(columns=['stats', 'types', 'abilities'])

df = df_pokemon.merge(df_species, on='species_id', how='left', validate='many_to_one')

# Needs to be last step, otherwise filters won't work
df = df.set_index('id')

print(f"N of Rows: {len(df)}")
df.head()

N of Rows: 1351


,name,is_default,weight,moves,species_id,hp,attack,defense,special_attack,special_defense,speed,type_1,type_2,ability_1,ability_2,hidden_ability,is_legendary,is_mythical,generation,is_fully_evolved
id,,,,,,,,,,,,,,,,,,,,
1,bulbasaur,True,69,"[swords-dance, vine-whip, tackle, body-slam, t...",1,45,49,49,65,65,45,grass,poison,overgrow,NaN,chlorophyll,False,False,1,False
2,ivysaur,True,130,"[swords-dance, vine-whip, tackle, body-slam, t...",2,60,62,63,80,80,60,grass,poison,overgrow,NaN,chlorophyll,False,False,1,False
3,venusaur,True,1000,"[swords-dance, vine-whip, tackle, body-slam, t...",3,80,82,83,100,100,80,grass,poison,overgrow,NaN,chlorophyll,False,False,1,True
4,charmander,True,85,"[fire-punch, thunder-punch, scratch, swords-da...",4,39,52,43,60,50,65,fire,NaN,blaze,NaN,solar-power,False,False,1,False
5,charmeleon,True,190,"[fire-punch, thunder-punch, scratch, swords-da...",5,58,64,58,80,65,80,fire,NaN,blaze,NaN,solar-power,False,False,1,False


In [68]:
# Data quality checks
print(f"Duplicated ids:   {df.index.duplicated().sum()}")
print(f"Duplicated names: {df['name'].duplicated().sum()}")
print(f"Duplicated rows:  {df.drop(columns='moves').duplicated().sum()}")

print("----------\nNulls per column:")
print(df.isna().sum())
print(f"Without abilities: {df.loc[df['ability_1'].isna(), 'name'].to_list()}")

valid_types = {'normal', 'fighting', 'flying', 'poison', 'ground', 'rock', 'bug', 'ghost', 'steel', 'fire', 'water', 'grass', 'electric', 'psychic', 'ice', 'dragon', 'dark', 'fairy'}
empty_moves = df['moves'].map(len).eq(0)

Duplicated ids:   0
Duplicated names: 0
Duplicated rows:  0
----------
Nulls per column:
name                  0
is_default            0
weight                0
moves                 0
species_id            0
hp                    0
attack                0
defense               0
special_attack        0
special_defense       0
speed                 0
type_1                0
type_2              586
ability_1             9
ability_2           738
hidden_ability      363
is_legendary          0
is_mythical           0
generation            0
is_fully_evolved      0
dtype: int64
Without abilities: ['zygarde-mega', 'heatran-mega', 'darkrai-mega', 'magearna-mega', 'magearna-original-mega', 'zeraora-mega', 'tatsugiri-curly-mega', 'tatsugiri-droopy-mega', 'tatsugiri-stretchy-mega']


In [69]:
print(f"Base stats out of 1-255:    {(~df[stat_cols].apply(lambda c: c.between(1, 255))).sum().sum()}")
print(f"Weight <= 0:                {df['weight'].le(0).sum()} -> {df.loc[df['weight'].le(0), 'name'].to_list()}")
print(f"Types outside the 18 types: {(df[type_cols].notna() & ~df[type_cols].isin(valid_types)).sum().sum()}")
print(f"type_1 == type_2:           {(df['type_1'] == df['type_2']).sum()}")
print(f"Empty moves:                {empty_moves.sum()} ({(empty_moves & df['is_default']).sum()} default forms)")

# Only the regional form of these species evolves
regional_only_evolutions = ['corsola', 'farfetchd', 'mr-mime', 'linoone', 'qwilfish', 'basculin-red-striped', 'basculin-blue-striped']

# These forms can't evolve, so Eviolite doesn't work on them
non_evolving_forms = [
    'pikachu-original-cap', 'pikachu-hoenn-cap', 'pikachu-sinnoh-cap', 'pikachu-unova-cap', 'pikachu-kalos-cap', 'pikachu-alola-cap', 'pikachu-partner-cap', 'pikachu-world-cap',
    'pikachu-rock-star', 'pikachu-belle', 'pikachu-pop-star', 'pikachu-phd', 'pikachu-libre', 'pikachu-cosplay',
    'pikachu-starter', 'eevee-starter', 'floette-eternal', 'floette-mega', 'gimmighoul-roaming',
    'pikachu-gmax', 'meowth-gmax', 'eevee-gmax'
]

print("----------")
print(f"Legendary: {df['is_legendary'].sum()} | Mythical: {df['is_mythical'].sum()} | Both: {(df['is_legendary'] & df['is_mythical']).sum()}")
print(f"Fully evolved: {df['is_fully_evolved'].sum()}")
print(df.loc[df['name'].isin(regional_only_evolutions), ['name', 'is_fully_evolved']])
print(df.loc[df['name'].isin(non_evolving_forms), ['name', 'is_fully_evolved']])

Base stats out of 1-255:    0
Weight <= 0:                1 -> ['eternatus-eternamax']
Types outside the 18 types: 0
type_1 == type_2:           0
Empty moves:                484 (292 default forms)
----------
Legendary: 120 | Mythical: 38 | Both: 0
Fully evolved: 840
                        name  is_fully_evolved
id                                            
83                 farfetchd             False
122                  mr-mime             False
211                 qwilfish             False
222                  corsola             False
264                  linoone             False
550     basculin-red-striped             False
10016  basculin-blue-striped             False
                       name  is_fully_evolved
id                                           
10061       floette-eternal             False
10080     pikachu-rock-star             False
10081         pikachu-belle             False
10082      pikachu-pop-star             False
10083           pikachu-phd     

In [70]:
# Eternamax (0) and Gigantamax (10000) weights are placeholders: the games show "????"
df['weight'] = df['weight'].mask(df['weight'].isin([0, 10000]))

# Their base form does not evolve, only the regional form does
df.loc[df['name'].isin(regional_only_evolutions), 'is_fully_evolved'] = True

# These forms can't evolve
df.loc[df['name'].isin(non_evolving_forms), 'is_fully_evolved'] = True

print(f"Null weight: {df['weight'].isna().sum()} | Fully evolved: {df['is_fully_evolved'].sum()}")
print(df.loc[df['name'].isin(regional_only_evolutions), ['name', 'is_fully_evolved']])
print(df.loc[df['name'].isin(non_evolving_forms), ['name', 'is_fully_evolved']])

Null weight: 35 | Fully evolved: 869
                        name  is_fully_evolved
id                                            
83                 farfetchd              True
122                  mr-mime              True
211                 qwilfish              True
222                  corsola              True
264                  linoone              True
550     basculin-red-striped              True
10016  basculin-blue-striped              True
                       name  is_fully_evolved
id                                           
10061       floette-eternal              True
10080     pikachu-rock-star              True
10081         pikachu-belle              True
10082      pikachu-pop-star              True
10083           pikachu-phd              True
10084         pikachu-libre              True
10085       pikachu-cosplay              True
10094  pikachu-original-cap              True
10095     pikachu-hoenn-cap              True
10096    pikachu-sinnoh-cap       

In [71]:
# Data types:
ability_cols = ['ability_1', 'ability_2', 'hidden_ability']

df[stat_cols] = df[stat_cols].astype('int16')
df['weight'] = df['weight'].astype('Int16')
df['species_id'] = df['species_id'].astype('int16')
df['generation'] = df['generation'].astype('int8')
df[type_cols + ability_cols] = df[type_cols + ability_cols].astype('str')

print(df.dtypes)
print(f"----------\nIndex dtype: {df.index.dtype}")
print(f"Names not in kebab-case: {(~df['name'].str.fullmatch(r'[a-z0-9-]+')).sum()}")

name                   str
is_default            bool
weight               Int16
moves               object
species_id           int16
hp                   int16
attack               int16
defense              int16
special_attack       int16
special_defense      int16
speed                int16
type_1                 str
type_2                 str
ability_1              str
ability_2              str
hidden_ability         str
is_legendary          bool
is_mythical           bool
generation            int8
is_fully_evolved      bool
dtype: object
----------
Index dtype: int64
Names not in kebab-case: 0


In [72]:
df[stat_cols].describe()

,hp,attack,defense,special_attack,special_defense,speed
count,1351.000000,1351.000000,1351.000000,1351.000000,1351.000000,1351.000000
mean,71.673575,82.757957,76.193930,75.418209,74.115470,72.000000
std,26.827529,32.351906,30.712902,34.026272,27.800826,30.642043
min,1.000000,5.000000,5.000000,10.000000,20.000000,5.000000
25%,55.000000,60.000000,55.000000,50.000000,54.000000,49.000000
50%,70.000000,80.000000,70.000000,69.000000,70.000000,70.000000
75%,85.000000,102.500000,95.000000,97.000000,90.000000,95.000000
max,255.000000,190.000000,250.000000,216.000000,250.000000,200.000000


In [73]:
# Highest base stat totals among default forms (the total is not stored)
df.loc[df['is_default'], ['name'] + stat_cols].assign(total=df[stat_cols].sum(axis=1)).nlargest(10, 'total')

,name,hp,attack,defense,special_attack,special_defense,speed,total
id,,,,,,,,
493,arceus,120,120,120,120,120,120,720
890,eternatus,140,85,95,145,95,130,690
150,mewtwo,106,110,90,154,90,130,680
249,lugia,106,90,130,90,154,110,680
250,ho-oh,106,130,90,110,154,90,680
384,rayquaza,105,150,90,150,90,95,680
483,dialga,100,120,120,150,100,90,680
484,palkia,90,120,100,150,120,100,680
487,giratina-altered,150,100,120,100,120,90,680


In [74]:
pd.crosstab(df['type_1'], df['type_2'].fillna('none'))

type_2,bug,dark,dragon,electric,fairy,fighting,fire,flying,ghost,grass,ground,ice,none,normal,poison,psychic,rock,steel,water
type_1,,,,,,,,,,,,,,,,,,,
bug,0,1,0,5,3,5,2,15,1,6,2,0,23,0,13,4,4,8,3
dark,0,0,4,0,4,3,4,7,3,3,1,3,17,6,2,3,0,3,0
dragon,0,1,0,1,1,3,1,7,3,0,10,7,14,1,0,4,0,0,6
electric,0,2,8,0,2,2,1,8,1,3,1,2,56,2,5,1,0,5,1
fairy,0,0,0,0,0,1,0,5,0,0,0,0,24,0,0,1,0,4,0
fighting,0,3,5,1,0,0,1,4,2,0,0,2,32,0,2,3,0,4,3
fire,3,1,3,0,0,8,0,8,6,0,3,0,38,3,1,4,3,2,1
flying,0,1,2,0,0,1,0,0,0,0,0,0,4,0,0,0,0,2,3
ghost,0,1,2,0,5,0,4,3,0,11,2,0,19,0,5,0,0,0,0


In [75]:
df['generation'].value_counts().sort_index()

generation
1    238
2    115
3    167
4    129
5    185
6    104
7    131
8    133
9    149
Name: count, dtype: int64

In [76]:
# Size of the Scarlet/Violet learnsets
df['moves'].map(len).describe()

count    1351.000000
mean       35.986677
std        29.463446
min         0.000000
25%         0.000000
50%        47.000000
75%        59.000000
max       234.000000
Name: moves, dtype: float64

In [77]:
# Lightest and heaviest
df.dropna(subset='weight').sort_values('weight')[['name', 'weight']].iloc[[0, 1, 2, -3, -2, -1]]

,name,weight
id,,
669,flabebe,1
93,haunter,1
798,kartana,1
10078,groudon-primal,9997
797,celesteela,9999
790,cosmoem,9999


In [78]:
SILVER_POKEAPI_PATH.mkdir(parents=True, exist_ok=True)

df.reset_index().to_parquet(SILVER_POKEAPI_PATH / 'pokemon.parquet', index=False)

# pyarrow reads the lists back as numpy arrays, so they are converted to lists before comparing
check = pd.read_parquet(SILVER_POKEAPI_PATH / 'pokemon.parquet').set_index('id')
check['moves'] = check['moves'].map(list)
print(f"Rows: {len(check)} | Same dtypes: {check.dtypes.equals(df.dtypes)} | Same data: {check.equals(df)}")

Rows: 1351 | Same dtypes: True | Same data: True
